In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from MODELS.pipeline import *
from MODELS.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 1


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

### Update projected starting lineups

In [4]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated c:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\MODELS/teamInfo.py
Updated 24 teams with confirmed lineups


### Top EVs for single bets

In [5]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['ODDS'] <= 250) & (usData['ODDS'] >= -250)]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
                             edge_threshold=0.20, stake=10, 
                             variance_inflation=1.1, distribution_type='skewnorm', 
                             use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'EXPECTED ROI', 'KELLY_FRACTION','SIGMA FLAG']].head(10)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,EXPECTED ROI,KELLY_FRACTION,SIGMA FLAG
0,Stephon Castle,Bovada,19.5,22.32,Over,200,1,9.25,92.5,0.462,High
1,Caris LeVert,BetOnline.ag,16.5,7.30,Under,-108,1,8.47,84.7,0.914,Med
2,Nikola Jokic,Bovada,23.5,20.83,Under,190,1,8.18,81.8,0.431,High
3,Caris LeVert,BetRivers,17.5,7.30,Under,-115,1,8.18,81.8,0.941,Med
4,Caris LeVert,DraftKings,16.5,7.30,Under,-113,1,8.07,80.7,0.912,Med


## Top EVs for 2 leg bets

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=10, 
                     variance_inflation=1.1, distribution_type='skewnorm',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

,NAME 1,NAME 2,LINE 1,LINE 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Landry Shamet,John Collins,4.5,16.5,over,under,1,7.64,0.382,Med,Med
1,Stephon Castle,John Collins,16.5,16.5,over,under,1,7.46,0.373,High,Med
2,OG Anunoby,John Collins,15.5,16.5,over,under,1,7.07,0.353,High,Med
3,Luguentz Dort,John Collins,9.5,16.5,under,under,1,6.96,0.348,Med,Med
4,Alex Sarr,John Collins,16.5,16.5,over,under,1,6.59,0.329,High,Med


### Prizepicks picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=10, 
                     variance_inflation=1.1, distribution_type='skewnorm',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

,NAME 1,NAME 2,LINE 1,LINE 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Caris LeVert,Victor Wembanyama,16.5,18.5,under,over,1,13.46,0.673,Med,High
1,Caris LeVert,John Collins,16.5,16.5,under,under,1,11.76,0.588,Med,Med
2,Caris LeVert,Jaylen Brown,16.5,20.5,under,over,1,10.98,0.549,Med,High
3,Victor Wembanyama,John Collins,18.5,16.5,over,under,1,10.56,0.528,High,Med
4,Duncan Robinson,Victor Wembanyama,15.5,18.5,under,over,1,10.31,0.516,Med,High


## 3 leg parlay

### Underdog picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.10, stake=10, 
                     variance_inflation=1.1, distribution_type='skewnorm', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3','MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Landry Shamet,Stephon Castle,John Collins,4.5,16.5,16.5,over,over,under,1,14.44,0.289,Med,High,Med
1,OG Anunoby,Landry Shamet,John Collins,15.5,4.5,16.5,over,over,under,1,13.88,0.278,High,Med,Med
2,Landry Shamet,Luguentz Dort,John Collins,4.5,9.5,16.5,over,under,under,1,13.73,0.275,Med,Med,Med
3,OG Anunoby,Stephon Castle,John Collins,15.5,16.5,16.5,over,over,under,1,13.64,0.273,High,High,Med
4,Stephon Castle,Luguentz Dort,John Collins,16.5,9.5,16.5,over,under,under,1,13.49,0.270,High,Med,Med


### Prizepicks picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.40, stake=100, 
                     variance_inflation=1.1, distribution_type='skewnorm', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Landry Shamet,Stephon Castle,John Collins,4.5,16.5,16.5,over,over,under,1,14.44,0.289,Med,High,Med
1,OG Anunoby,Landry Shamet,John Collins,15.5,4.5,16.5,over,over,under,1,13.88,0.278,High,Med,Med
2,Landry Shamet,Luguentz Dort,John Collins,4.5,9.5,16.5,over,under,under,1,13.73,0.275,Med,Med,Med
3,OG Anunoby,Stephon Castle,John Collins,15.5,16.5,16.5,over,over,under,1,13.64,0.273,High,High,Med
4,Stephon Castle,Luguentz Dort,John Collins,16.5,9.5,16.5,over,under,under,1,13.49,0.270,High,Med,Med
